# Thực nghiệm và Báo cáo Đánh giá: Chandra (Datalab Layout OCR)

Notebook này cung cấp quy trình khép kín để:
1. **Tự động kiểm tra môi trường và trọng số** `Chandra` (Datalab OCR / Document Intelligence).
2. **Thực hiện suy luận (Inference)** trên tập dữ liệu DocLayNet.
3. **Tính toán chỉ số định lượng** (mAP, Precision, Recall, F1).
4. **Trực quan hóa so sánh** giữa Ground Truth và dự đoán của mô hình.

In [ ]:
# 1. Khởi tạo đường dẫn và cấu hình hệ thống
import os
import sys
import json
import subprocess
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

NOTEBOOK_DIR = Path.cwd()
METHOD_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR / 'methods' / 'chandra'
PROJECT_ROOT = METHOD_DIR.parent.parent

print(f"Project root: {PROJECT_ROOT}")
print(f"Method dir:   {METHOD_DIR}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(METHOD_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(METHOD_DIR / 'src'))

In [ ]:
# 2. Kiểm tra trọng số và tự động tải nếu cần thiết
weights_dir = METHOD_DIR / "weights"
weights_dir.mkdir(parents=True, exist_ok=True)

has_weights = any(weights_dir.glob("*.safetensors")) and (weights_dir / "config.json").exists()
if has_weights:
    print(f"[V] Đã tìm thấy trọng số đầy đủ tại: {weights_dir}")
else:
    print(f"[!] Trọng số chưa tải hoặc chưa đầy đủ. Đang tải từ Hugging Face Hub (datalab-to/chandra-ocr-2)...\n")
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id="datalab-to/chandra-ocr-2",
        local_dir=str(weights_dir),
        ignore_patterns=["*.msgpack", "*.h5", "*.ot"]
    )
    print(f"[V] Đã tải xong trọng số về {weights_dir}!")

In [ ]:
# 3. Chạy suy luận trên tập mẫu DocLayNet
inference_script = METHOD_DIR / "src" / "run_inference.py"
cmd_infer = [
    sys.executable, str(inference_script),
    "--dataset", "doclaynet",
    "--limit", "20"
]
print("Đang chạy suy luận với lệnh:", " ".join(cmd_infer))
res = subprocess.run(cmd_infer, capture_output=True, text=True, cwd=str(PROJECT_ROOT))
print(res.stdout)
if res.returncode != 0:
    print("Lỗi:", res.stderr)

In [ ]:
# 4. Đánh giá định lượng kết quả
eval_script = PROJECT_ROOT / "scripts" / "evaluate_predictions.py"
chart_out = PROJECT_ROOT / "outputs" / "reports" / "chandra_doclaynet" / "metrics_chart.png"
chart_out.parent.mkdir(parents=True, exist_ok=True)

cmd_eval = [
    sys.executable, str(eval_script),
    "--method", "chandra",
    "--dataset", "doclaynet",
    "--allow-incomplete",
    "--loose",
    "--plot", str(chart_out)
]
print("Đang đánh giá kết quả...")
res_eval = subprocess.run(cmd_eval, capture_output=True, text=True, cwd=str(PROJECT_ROOT))
print(res_eval.stdout)
if res_eval.returncode != 0:
    print("Cảnh báo:", res_eval.stderr)

In [ ]:
# 5. Hiển thị biểu đồ Metrics
if chart_out.exists():
    plt.figure(figsize=(10, 6), dpi=120)
    img = Image.open(chart_out)
    plt.imshow(img)
    plt.axis('off')
    plt.title("Đánh giá hiệu năng Chandra trên DocLayNet", fontsize=14, fontweight='bold')
    plt.show()
else:
    print("Chưa có biểu đồ metrics.")

In [ ]:
# 6. Trực quan hóa Bounding Box so sánh với Ground Truth
vis_script = PROJECT_ROOT / "visualization" / "scripts" / "visualize_predictions.py"
vis_out_dir = PROJECT_ROOT / "outputs" / "reports" / "chandra_doclaynet"

cmd_vis = [
    sys.executable, str(vis_script),
    "--method", "chandra",
    "--dataset", "doclaynet",
    "--limit", "5",
    "--output-dir", str(vis_out_dir)
]
print("Đang xuất ảnh trực quan hóa...")
res_vis = subprocess.run(cmd_vis, capture_output=True, text=True, cwd=str(PROJECT_ROOT))
print(res_vis.stdout)
if res_vis.returncode != 0:
    print("Cảnh báo:", res_vis.stderr)

In [ ]:
# 7. Hiển thị ảnh trực quan hóa mẫu
vis_files = list(vis_out_dir.glob("*.png"))
sample_images = [f for f in vis_files if f.name != "metrics_chart.png"][:2]

for f in sample_images:
    plt.figure(figsize=(12, 12), dpi=100)
    plt.imshow(Image.open(f))
    plt.axis('off')
    plt.title(f.name, fontsize=12)
    plt.show()